# Case Study 02 — Panel Construction

**Erick Condoy** · Economist (UNL) · Quant Researcher  
Repository: [credit-risk-lab](https://github.com/EryckFCS/credit-risk-lab)

---

## Objective

Build a **balanced panel** `institution × period` with:
- PD proxy target (delinquency deterioration rate)
- lagged supervisory features
- macroeconomic covariates
- stationarity checks and transformation decisions

This panel feeds notebooks 03 (EDA) and 04 (model).

| Section | Content |
|---|---|
| 1 | Load canonical data from NB01 |
| 2 | Panel skeleton and balance check |
| 3 | PD proxy target construction |
| 4 | Supervisory feature engineering (lags, growth rates, ratios) |
| 5 | Macroeconomic merge |
| 6 | Stationarity checks (ADF / KPSS) |
| 7 | Missing data audit and forward-fill policy |
| 8 | Final panel export |


In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)

BASE  = Path('../')
PROC  = BASE / 'data' / 'processed'
PANEL = BASE / 'data' / 'panel'
PANEL.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#dcd9d5', 'axes.labelcolor': '#28251d',
    'xtick.color': '#7a7974', 'ytick.color': '#7a7974',
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'semibold', 'figure.dpi': 120,
})
TEAL = '#01696f'; MAROON = '#a12c7b'; GRAY = '#bab9b4'
GREEN = '#437a22'; ORANGE = '#964219'; GOLD = '#d19900'
print('Environment ready.')

## 1. Load Canonical Data

In [ ]:
df_long = pd.read_parquet(PROC / 'sbs_long_template.parquet')
df_long['period'] = pd.to_datetime(df_long['period'])

print('Schema:')
print(df_long.dtypes.to_string())
print()
print(f'Rows     : {len(df_long):,}')
print(f'Entities : {df_long.institution_id.nunique()}')
print(f'Period   : {df_long.period.min().date()} — {df_long.period.max().date()}')
print(f'Variables: {sorted(df_long.variable.unique())}')

## 2. Panel Skeleton and Balance Check

In [ ]:
df_wide = df_long.pivot_table(
    index=['institution_id', 'period'],
    columns='variable',
    values='value',
    aggfunc='first',
).reset_index()
df_wide.columns.name = None

counts = df_wide.groupby('institution_id')['period'].count()
total_periods = df_wide['period'].nunique()
balance_rate  = (counts == total_periods).mean()

print(f'Total institutions : {df_wide.institution_id.nunique()}')
print(f'Total periods      : {total_periods}')
print(f'Panel balance rate : {balance_rate:.1%}')
display(counts.to_frame('n_periods'))

## 3. PD Proxy Target Construction

PD proxy = first difference of NPL ratio, capturing credit quality deterioration:

$$\Delta\text{NPL}_{i,t} = \text{NPL}_{i,t} - \text{NPL}_{i,t-1}$$

Binary target calibrated at p75 of positive deltas — material deterioration event,
equivalent to IFRS 9 Stage 1 → Stage 2 migration trigger.

In [ ]:
df_wide = df_wide.sort_values(['institution_id', 'period']).reset_index(drop=True)

df_wide['npl_lag1'] = df_wide.groupby('institution_id')['npl_ratio'].shift(1)
df_wide['delta_npl'] = df_wide['npl_ratio'] - df_wide['npl_lag1']

positive_deltas = df_wide.loc[df_wide['delta_npl'] > 0, 'delta_npl']
delta_threshold = positive_deltas.quantile(0.75) if len(positive_deltas) >= 4 else 0.005

df_wide['y_deterioration'] = (df_wide['delta_npl'] > delta_threshold).astype(int)

print(f'Delta NPL threshold (p75)  : {delta_threshold:.4f}')
print(f'Deterioration rate (target): {df_wide.y_deterioration.mean():.2%}')
print(df_wide['y_deterioration'].value_counts().to_string())

## 4. Supervisory Feature Engineering

In [ ]:
g = df_wide.groupby('institution_id')

for lag in [1, 2, 3]:
    df_wide[f'npl_lag{lag}'] = g['npl_ratio'].shift(lag)

df_wide['coverage_lag1']  = g['coverage_ratio'].shift(1)
df_wide['delta_coverage'] = df_wide['coverage_ratio'] - df_wide['coverage_lag1']
df_wide['portfolio_growth']  = g['gross_portfolio'].pct_change(1)
df_wide['impaired_growth']   = g['impaired_portfolio'].pct_change(1)
df_wide['npl_vol3']   = g['npl_ratio'].transform(lambda x: x.rolling(3, min_periods=2).std())
df_wide['npl_vs_mean'] = df_wide['npl_ratio'] - g['npl_ratio'].transform('mean')

SUPERVISORY_FEATURES = [
    'npl_lag1', 'npl_lag2', 'npl_lag3',
    'coverage_lag1', 'delta_coverage',
    'portfolio_growth', 'impaired_growth',
    'npl_vol3', 'npl_vs_mean',
]

display(df_wide[SUPERVISORY_FEATURES].describe().T.round(4))

## 5. Macroeconomic Merge (BCE — leakage-controlled)

In [ ]:
# Replace synthetic block with pd.read_csv/parquet of real BCE series
np.random.seed(42)
macro_periods = df_wide['period'].unique()
macro_df = pd.DataFrame({
    'period': macro_periods,
    'activity_growth': np.random.normal(0.018, 0.015, len(macro_periods)),
    'inflation': np.random.normal(0.025, 0.008, len(macro_periods)),
    'credit_system_growth': np.random.normal(0.085, 0.040, len(macro_periods)),
    'oil_price_change': np.random.normal(0.0, 0.12, len(macro_periods)),
})

MACRO_FEATURES = ['activity_growth', 'inflation', 'credit_system_growth', 'oil_price_change']

df_panel = df_wide.merge(macro_df, on='period', how='left')
for feat in MACRO_FEATURES:
    df_panel[f'{feat}_lag1'] = df_panel.groupby('institution_id')[feat].shift(1)

MACRO_LAG_FEATURES = [f'{f}_lag1' for f in MACRO_FEATURES]
print('Macro merged. Leakage-controlled features:', MACRO_LAG_FEATURES)

## 6. Stationarity Checks (ADF / KPSS)

In [ ]:
def stationarity_report(series, name):
    s = series.dropna()
    if len(s) < 8:
        return {'variable': name, 'n': len(s), 'ADF_p': np.nan, 'KPSS_p': np.nan, 'conclusion': 'insufficient data'}
    adf_p = adfuller(s, autolag='AIC')[1]
    try:
        kpss_p = kpss(s, regression='c', nlags='auto')[1]
    except:
        kpss_p = np.nan
    if adf_p < 0.05 and (np.isnan(kpss_p) or kpss_p > 0.05):
        conclusion = 'STATIONARY'
    elif adf_p >= 0.05:
        conclusion = 'NON-STATIONARY → first-difference'
    else:
        conclusion = 'CONFLICTING → inspect plot'
    return {'variable': name, 'n': len(s), 'ADF_p': round(adf_p,4),
            'KPSS_p': round(kpss_p,4) if not np.isnan(kpss_p) else np.nan, 'conclusion': conclusion}

ALL_FEATURES = SUPERVISORY_FEATURES + MACRO_LAG_FEATURES
stat_df = pd.DataFrame([stationarity_report(df_panel[v].dropna(), v)
                         for v in ALL_FEATURES if v in df_panel.columns])

def color_conclusion(val):
    if 'STATIONARY' in str(val) and 'NON' not in str(val): return 'background-color:#d4dfcc'
    elif 'NON' in str(val): return 'background-color:#ddcfc6'
    return ''

display(stat_df.style.applymap(color_conclusion, subset=['conclusion']))

## 7. Missing Data Audit and Fill Policy

In [ ]:
for feat in MACRO_LAG_FEATURES:
    df_panel[feat] = df_panel.groupby('institution_id')[feat].ffill(limit=2)

df_model = df_panel.dropna(subset=ALL_FEATURES + ['y_deterioration']).copy().reset_index(drop=True)

print(f'Before drop: {len(df_panel):,} rows')
print(f'After drop : {len(df_model):,} rows  ({1-len(df_model)/len(df_panel):.1%} removed)')

miss = df_panel[ALL_FEATURES].isna().mean().sort_values(ascending=False)
miss = miss[miss > 0]
if miss.empty: print('No missing in model features after fill policy.')
else: display(miss.to_frame('missing_pct').style.format('{:.2%}'))

## 8. Final Panel Export

In [ ]:
import json, datetime

panel_meta = {
    'created': str(datetime.date.today()),
    'n_rows': int(len(df_model)),
    'n_entities': int(df_model.institution_id.nunique()),
    'n_periods': int(df_model.period.nunique()),
    'period_min': str(df_model.period.min().date()),
    'period_max': str(df_model.period.max().date()),
    'target_rate': round(float(df_model.y_deterioration.mean()), 4),
    'supervisory_features': SUPERVISORY_FEATURES,
    'macro_features': MACRO_LAG_FEATURES,
    'all_features': ALL_FEATURES,
    'target': 'y_deterioration',
    'target_definition': 'delta_npl > p75(positive deltas) — IFRS9 Stage1→2 proxy',
}

df_model.to_parquet(PANEL / 'panel_model_ready.parquet', index=False)
with open(PANEL / 'panel_metadata.json', 'w') as f:
    json.dump(panel_meta, f, indent=2)

print('[saved] data/panel/panel_model_ready.parquet')
print('[saved] data/panel/panel_metadata.json')
print()
for k, v in panel_meta.items():
    if k not in ['supervisory_features','macro_features','all_features']:
        print(f'  {k:<28} {v}')
print(f'  supervisory_features ({len(SUPERVISORY_FEATURES)}): {SUPERVISORY_FEATURES}')
print(f'  macro_features       ({len(MACRO_LAG_FEATURES)}): {MACRO_LAG_FEATURES}')
print()
print('Next → 03_eda_macro_credit.ipynb')